In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D6 — Microsoft FY24 Q1 Press Release
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip -q install pymupdf pymupdf4llm

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import re
import fitz
import pymupdf4llm

DOCUMENT_ID = "D6"
DOCUMENT_NAME = "Microsoft FY24 Q1 Press Release"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "pymupdf4llm page-aware Markdown conversion "
    "with complete-document retention"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "b3cf121cac62f6807d41e4e013a6edeebec0cbbf4f0ea2995e1fcc135abaac8d"

EXPECTED_PAGE_COUNT = 10
EXPECTED_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

EXPECTED_PART_COUNTS = {
    1: 46,
    2: 25,
    3: 34,
    4: 34,
    5: 8
}

EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]

NUMERIC_OR_NULL_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

PART_PAGE_RANGES = {
    1: [1, 2, 3],
    2: [6, 7],
    3: [8],
    4: [9],
    5: [10]
}

OUTPUT_DIR = Path("outputs_D6_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Fixed Stage 1 records:", EXPECTED_RECORD_COUNT)



In [ ]:
# ============================================================
# 1. Upload and verify original D6 PDF
# ============================================================

uploaded = files.upload()

pdf_files = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError("Upload exactly one original D6 PDF.")

SOURCE_PATH = pdf_files[0]

def sha256_file(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D6 source format.")

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded D6 PDF does not match the frozen source identity.")

pdf_document = fitz.open(SOURCE_PATH)
PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = PAGE_COUNT == EXPECTED_PAGE_COUNT

if not PAGE_COUNT_VALID:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

page_character_counts = [
    len(page.get_text("text").strip())
    for page in pdf_document
]

TEXT_EXTRACTABLE = all(count > 0 for count in page_character_counts)

if not TEXT_EXTRACTABLE:
    raise ValueError(
        "D6 is expected to contain a machine-readable text layer. "
        "OCR is not introduced in Branch B."
    )

print("Source SHA-256:", SOURCE_SHA256)
print("Page count:", PAGE_COUNT)
print("Machine-readable text layer:", TEXT_EXTRACTABLE)


In [ ]:
# ============================================================
# 2. Verify expected document components
# ============================================================

full_source_text = "\n".join(
    page.get_text("text")
    for page in pdf_document
)

EXPECTED_COMPONENT_MARKERS = {
    "quarterly_results":
        "Revenue was $56.5 billion",
    "business_highlights":
        "Business Highlights",
    "financial_performance_reconciliation":
        "Financial Performance Constant Currency Reconciliation",
    "segment_revenue_reconciliation":
        "Segment Revenue Constant Currency Reconciliation",
    "selected_product_reconciliation":
        "Selected Product and Service Revenue Constant Currency Reconciliation",
    "income_statements":
        "INCOME STATEMENTS",
    "comprehensive_income_statements":
        "COMPREHENSIVE INCOME STATEMENTS",
    "balance_sheets":
        "BALANCE SHEETS",
    "cash_flows":
        "CASH FLOWS STATEMENTS",
    "segment_revenue_operating_income":
        "SEGMENT REVENUE AND OPERATING INCOME"
}

component_checks = {
    key: marker.casefold() in full_source_text.casefold()
    for key, marker in EXPECTED_COMPONENT_MARKERS.items()
}

DOCUMENT_GROUNDING_VALID = all(component_checks.values())

print(json.dumps(component_checks, indent=2))

if not DOCUMENT_GROUNDING_VALID:
    raise ValueError("Expected D6 source components were not all found.")


In [ ]:
# ============================================================
# 3. Conversion of the PDF to page-aware Markdown
# ============================================================

conversion_error = None
page_chunks = None

try:
    page_chunks = pymupdf4llm.to_markdown(
        str(SOURCE_PATH),
        page_chunks=True,
        write_images=False,
        show_progress=True
    )
except Exception as exc:
    conversion_error = str(exc)

if conversion_error is not None:
    raise RuntimeError(
        "D6 Branch B structural conversion failed. "
        "No fallback representation is used because the conversion "
        "method must remain explicit and reproducible.\n"
        f"Original error: {conversion_error}"
    )

if not isinstance(page_chunks, list):
    raise TypeError("Expected pymupdf4llm page_chunks=True to return a list.")

if len(page_chunks) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} converted page chunks; "
        f"observed {len(page_chunks)}."
    )

page_markdown = {}

for page_number, chunk in enumerate(page_chunks, start=1):
    if isinstance(chunk, dict):
        text = chunk.get("text", "")
    else:
        text = str(chunk)

    if not text.strip():
        raise ValueError(
            f"Converted Markdown for source page {page_number} is empty."
        )

    page_markdown[page_number] = text.rstrip()

markdown_sections = [
    "# D6 — Microsoft FY24 Q1 Press Release",
    "",
    "> Complete structural conversion of the original 10-page PDF.",
    "> The same complete representation is used in all five extraction parts.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    markdown_sections += [
        f"## Source Page {page_number}",
        "",
        page_markdown[page_number],
        ""
    ]

STRUCTURAL_MARKDOWN = "\n".join(markdown_sections).rstrip() + "\n"

STRUCTURAL_MARKDOWN_PATH = (
    OUTPUT_DIR / "D6_branch_B_structural_markdown.md"
)

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)

print("Structural Markdown saved:", STRUCTURAL_MARKDOWN_PATH)
print("Representation SHA-256:", STRUCTURAL_MARKDOWN_SHA256)
print("Converted pages:", len(page_markdown))


In [ ]:
# ============================================================
# 4. Conversion-integrity checks
# ============================================================

markdown_lower = STRUCTURAL_MARKDOWN.casefold()

marker_checks = {
    key: marker.casefold() in markdown_lower
    for key, marker in EXPECTED_COMPONENT_MARKERS.items()
}

page_boundary_checks = {
    str(page_number):
        f"## Source Page {page_number}" in STRUCTURAL_MARKDOWN
    for page_number in range(1, EXPECTED_PAGE_COUNT + 1)
}


REPRESENTATIVE_CONTENT = [
    "$56.5 billion",
    "$31.8 billion",
    "50,122",
    "56,517",
    "22,291",
    "445,785",
    "80,452",
    "18,592",
    "11,751"
]

representative_content_checks = {
    value: value.casefold() in markdown_lower
    for value in REPRESENTATIVE_CONTENT
}

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": len(page_markdown),
    "all_source_pages_retained":
        all(page_boundary_checks.values()),
    "page_boundary_checks": page_boundary_checks,
    "expected_component_checks": marker_checks,
    "all_expected_components_preserved":
        all(marker_checks.values()),
    "representative_content_checks":
        representative_content_checks,
    "all_representative_content_preserved":
        all(representative_content_checks.values()),
    "conversion_method":
      CONVERSION_METHOD,
    "conversion_fallback_used": False,
    "part_specific_source_filtering_applied": False,
    "same_complete_representation_used_for_all_parts": True,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "page_removal_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "manual_correction_applied": False,
    "normalisation_applied": False,
    "conversion_integrity_passed": all([
        SOURCE_HASH_MATCH,
        PAGE_COUNT == EXPECTED_PAGE_COUNT,
        len(page_markdown) == EXPECTED_PAGE_COUNT,
        all(page_boundary_checks.values()),
        all(marker_checks.values()),
        all(representative_content_checks.values())
    ])
}

CONVERSION_INTEGRITY_PATH = (
    OUTPUT_DIR / "D6_branch_B_conversion_integrity.json"
)

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    CONVERSION_INTEGRITY,
    ensure_ascii=False,
    indent=2
))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError("D6 Branch B conversion integrity failed.")


In [ ]:
# ============================================================
# 5. Preservation of representation metadata
# ============================================================

REPRESENTATION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Complete PDF converted to page-aware structural Markdown",
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_PATH.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": len(page_markdown),
    "structural_conversion_applied": True,
    "conversion_method": CONVERSION_METHOD,
    "conversion_fallback_used": False,
    "complete_source_document_retained": True,
    "page_boundaries_made_explicit": True,
    "part_specific_page_filtering_applied": False,
    "same_complete_representation_used_for_all_five_parts": True,
    "split_extraction_applied": True,
    "split_part_count": 5,
    "split_strategy":
        "Same predefined source-scope partition as Branch A",
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D6_branch_B_representation.json"
)

REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(REPRESENTATION, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 6. Five Branch B prompts using the frozen Branch A scopes
# ============================================================

COMMON_INSTRUCTIONS = r'''
You are an information extraction assistant.

Extract the requested financial and quantitative records represented
within the defined source scope of the attached structurally converted
Markdown representation of the Microsoft FY24 Q1 Press Release.

Treat the attached structurally converted Markdown document as the only
source of information.

The complete 10-page source representation is attached. Process only
the source scope defined for the current extraction part.

For every included record return exactly these twelve fields:

- Category
- Statement or Section
- Metric
- Business Area
- Value 2023
- Value 2022
- GAAP YoY Change
- Constant Currency Impact
- Constant Currency YoY Change
- Unit
- Reporting Period
- Source Location

General extraction rules:

- Use JSON numbers for explicitly represented numeric values.
- Use JSON null when a field is not explicitly represented.
- Preserve negative values.
- Preserve explicit zero values.
- Preserve the source measurement scale.
- Preserve repeated observations when they occur in distinct source
  sections.
- Do not calculate or infer values.
- Do not derive prior-period values from growth rates.
- Do not convert units.
- Do not normalise measurement scales.
- Do not silently correct source values.
- Do not use external knowledge.
- Do not include publication metadata, webcast details, contact details,
  URLs, qualitative outlook text or forward-looking risk narrative.
- Do not include values that appear only inside descriptive accounting
  line-item labels when they are not primary observations.
- Ignore Markdown syntax and representation metadata that are not
  source financial observations.

For Balance Sheets only:

- place the September 30, 2023 amount in "Value 2023";
- place the June 30, 2023 amount in "Value 2022".

Return only valid JSON.

Do not include Markdown fences, explanations or commentary.

Return the result using exactly this top-level structure:

{
  "document_id": "D6",
  "branch": "B",
  "part": PART_NUMBER,
  "records": [
    {
      "Category": null,
      "Statement or Section": null,
      "Metric": null,
      "Business Area": null,
      "Value 2023": null,
      "Value 2022": null,
      "GAAP YoY Change": null,
      "Constant Currency Impact": null,
      "Constant Currency YoY Change": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
'''.strip()

PART_SCOPES = {
    1: r'''
PART 1 SOURCE SCOPE

Include only:

1. The quantitative company-performance observations represented in the
   quarterly-results and business-highlights sections on page 1 and the
   shareholder-return observation on page 2.

2. Every represented metric in the following three constant-currency
   reconciliation tables on page 3:

   - Financial Performance Constant Currency Reconciliation
   - Segment Revenue Constant Currency Reconciliation
   - Selected Product and Service Revenue Constant Currency Reconciliation

Valid Category values for this part:

- Narrative performance highlight
- Financial performance reconciliation
- Segment revenue reconciliation
- Selected product and service reconciliation

Valid Source Location values:

- Page 1 — Quarterly results
- Page 1 — Business Highlights
- Page 2 — Shareholder returns
- Page 3 — Financial Performance Constant Currency Reconciliation
- Page 3 — Segment Revenue Constant Currency Reconciliation
- Page 3 — Selected Product and Service Revenue Constant Currency Reconciliation

Extract every record represented within this defined source scope.
''',

    2: r'''
PART 2 SOURCE SCOPE

Include every primary financial line item represented in:

- Income Statements on page 6;
- Comprehensive Income Statements on page 7.

Exclude section headings and accounting-label values that are not
primary financial line-item observations.

Valid Category values:

- Income statement
- Comprehensive income statement

Valid Source Location values:

- Page 6 — Income Statements
- Page 7 — Comprehensive Income Statements

Use:

- "Income Statements" or "Comprehensive Income Statements" for
  Statement or Section;
- "Corporate" for Business Area;
- "Three months ended September 30" for Reporting Period.

Extract every record represented within this defined source scope.
''',

    3: r'''
PART 3 SOURCE SCOPE

Include every primary Balance Sheets line item represented on page 8.

Exclude:

- headings without amounts;
- Commitments and contingencies;
- allowance amounts embedded only in the accounts-receivable label;
- accumulated depreciation amounts embedded only in the property and
  equipment label;
- authorised and outstanding share quantities embedded only in the
  common-stock label.

Use:

- Category: "Balance sheet"
- Statement or Section: "Balance Sheets"
- Business Area: "Corporate"
- Unit: "USD millions"
- Reporting Period: "September 30, 2023 and June 30, 2023"
- Source Location: "Page 8 — Balance Sheets"

Place September 30, 2023 values in "Value 2023".

Place June 30, 2023 values in "Value 2022".

Extract every record represented within this defined source scope.
''',

    4: r'''
PART 4 SOURCE SCOPE

Include every primary Cash Flows Statements line item represented on
page 9.

Preserve parentheses as negative values and preserve explicit zeros.

Distinguish the two separate source observations:

- Other, net — financing
- Other, net — investing

Use:

- Category: "Cash flow statement"
- Statement or Section: "Cash Flows Statements"
- Business Area: "Corporate"
- Unit: "USD millions"
- Reporting Period: "Three months ended September 30"
- Source Location: "Page 9 — Cash Flows Statements"

Extract every record represented within this defined source scope.
''',

    5: r'''
PART 5 SOURCE SCOPE

Include every represented revenue and operating-income row from the
Segment Revenue and Operating Income table on page 10.

The table contains revenue observations and operating-income
observations for the represented business segments and total rows.

Use:

- Category: "Segment revenue and operating income"
- Statement or Section: "Segment Revenue and Operating Income"
- Unit: "USD millions"
- Reporting Period: "Three months ended September 30"
- Source Location: "Page 10 — Segment Revenue and Operating Income"

Extract every record represented within this defined source scope.
'''
}

PART_PROMPT_PATHS = {}

for part_number, part_scope in PART_SCOPES.items():
    prompt = (
        COMMON_INSTRUCTIONS.replace(
            "PART_NUMBER",
            str(part_number)
        )
        + "\n\n"
        + part_scope.strip()
    )

    path = (
        OUTPUT_DIR
        / f"D6_branch_B_prompt_part_{part_number}.txt"
    )

    path.write_text(
        prompt,
        encoding="utf-8"
    )

    PART_PROMPT_PATHS[part_number] = path

    print(
        f"Part {part_number}:",
        path.name,
        "| SHA-256:",
        sha256_file(path)
    )


In [ ]:
# ============================================================
# 7. Creation of pre-extraction experiment metadata
# ============================================================

PROMPT_FILE_METADATA = {
    str(part_number): {
        "file": path.name,
        "sha256": sha256_file(path)
    }
    for part_number, path
    in sorted(PART_PROMPT_PATHS.items())
}

EXPERIMENT_METADATA_PRE = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "source_structure": {
        "expected_page_count": EXPECTED_PAGE_COUNT,
        "observed_page_count": PAGE_COUNT,
        "page_count_verified": PAGE_COUNT_VALID,
        "machine_readable_text_layer": TEXT_EXTRACTABLE,
        "expected_components_verified": DOCUMENT_GROUNDING_VALID
    },
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "conversion_fallback_used": False,
    "complete_source_document_retained": True,
    "part_specific_page_filtering_applied": False,
    "same_complete_representation_used_for_all_parts": True,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "manual_correction_applied": False,
    "split_extraction_applied": True,
    "split_part_count": 5,
    "split_strategy":
        "Same deterministic source-section partition as Branch A",
    "same_split_required_across_branches": True,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_part_counts": EXPECTED_PART_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "content_validation_performed":
        False,
    "conversion_integrity_file":
        CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_files": PROMPT_FILE_METADATA,
    "expected_output_format":
        "Five JSON objects with document_id, branch, part and records",
    "execution_environment":
        "Five independent ChatGPT conversations",
}

EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D6_branch_B_experiment_metadata_pre.json"
)

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_METADATA_PRE,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 8. Download the Branch B representation and five prompts
# ============================================================

for path in [
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

for part_number in sorted(PART_PROMPT_PATHS):
    files.download(
        PART_PROMPT_PATHS[part_number]
    )

print(
    "\nExecution instructions:\n"
    "Run five independent ChatGPT conversations.\n"
    "In EVERY conversation upload the SAME "
    "D6_branch_B_structural_markdown.md.\n"
    "Then submit only the corresponding part prompt.\n"
    "Preserve the first complete response exactly as returned.\n"
    "Do not upload the original PDF or Stage 1 reference dataset."
)


In [ ]:
# ============================================================
# 9. Upload and preserve five raw Branch B responses
# ============================================================

uploaded_outputs = files.upload()

uploaded_txt_paths = [
    Path(name)
    for name in uploaded_outputs
    if name.lower().endswith(".txt")
]

if len(uploaded_txt_paths) != 5:
    raise ValueError(
        "Upload exactly five TXT raw-response files."
    )

def detect_part_number(path):
    match = re.search(
        r"part[_\- ]?([1-5])",
        path.stem,
        flags=re.IGNORECASE
    )

    if match is None:
        raise ValueError(
            f"Could not identify part number from {path.name}."
        )

    return int(match.group(1))

PART_RAW_RESPONSE_PATHS = {}

for uploaded_path in uploaded_txt_paths:
    part_number = detect_part_number(uploaded_path)

    if part_number in PART_RAW_RESPONSE_PATHS:
        raise ValueError(
            f"More than one response was detected for part {part_number}."
        )

    raw_text = uploaded_path.read_text(
        encoding="utf-8"
    )

    if not raw_text.strip():
        raise ValueError(
            f"Raw response for part {part_number} is empty."
        )

    preserved_path = (
        OUTPUT_DIR
        / f"D6_branch_B_raw_response_part_{part_number}.txt"
    )

    preserved_path.write_text(
        raw_text,
        encoding="utf-8"
    )

    PART_RAW_RESPONSE_PATHS[part_number] = preserved_path

if set(PART_RAW_RESPONSE_PATHS) != set(EXPECTED_PART_COUNTS):
    raise ValueError(
        "Uploaded files must represent parts 1, 2, 3, 4 and 5."
    )

PART_RAW_RESPONSE_METADATA = {
    str(part_number): {
        "file": path.name,
        "sha256": sha256_file(path),
        "character_count":
            len(path.read_text(encoding="utf-8"))
    }
    for part_number, path
    in sorted(PART_RAW_RESPONSE_PATHS.items())
}

print(json.dumps(
    PART_RAW_RESPONSE_METADATA,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 10. Parse five raw responses independently
# ============================================================

part_parsing_results = {}
part_records = {}
combined_records = []

all_parts_json_valid = True
all_parts_records_evaluable = True

for part_number in sorted(PART_RAW_RESPONSE_PATHS):
    raw_path = PART_RAW_RESPONSE_PATHS[part_number]
    raw_text = raw_path.read_text(encoding="utf-8")

    valid_json = False
    parsing_error = None
    parsed_part = None

    try:
        parsed_part = json.loads(raw_text)
        valid_json = True
    except json.JSONDecodeError as error:
        parsing_error = str(error)

    top_level_object_valid = (
        valid_json and isinstance(parsed_part, dict)
    )

    document_id_present = (
        top_level_object_valid
        and "document_id" in parsed_part
    )
    document_id_correct = (
        document_id_present
        and parsed_part.get("document_id") == DOCUMENT_ID
    )

    branch_present = (
        top_level_object_valid
        and "branch" in parsed_part
    )
    branch_correct = (
        branch_present
        and parsed_part.get("branch") == BRANCH
    )

    part_present = (
        top_level_object_valid
        and "part" in parsed_part
    )
    part_correct = (
        part_present
        and parsed_part.get("part") == part_number
    )

    records_present = (
        top_level_object_valid
        and "records" in parsed_part
    )
    records_is_list = (
        records_present
        and isinstance(parsed_part.get("records"), list)
    )

    records_evaluable = all([
        valid_json,
        top_level_object_valid,
        document_id_present,
        document_id_correct,
        branch_present,
        branch_correct,
        part_present,
        part_correct,
        records_present,
        records_is_list
    ])

    records = (
        parsed_part["records"]
        if records_evaluable
        else []
    )

    observed_part_count = (
        len(records)
        if records_evaluable
        else None
    )

    part_record_count_matches = (
        observed_part_count == EXPECTED_PART_COUNTS[part_number]
        if records_evaluable
        else None
    )

    part_records[part_number] = records

    part_parsing_results[part_number] = {
        "raw_response_file": raw_path.name,
        "raw_response_sha256": sha256_file(raw_path),
        "json_valid": valid_json,
        "json_parsing_error": parsing_error,
        "top_level_object_valid": top_level_object_valid,
        "document_id_present": document_id_present,
        "document_id_correct": document_id_correct,
        "branch_present": branch_present,
        "branch_correct": branch_correct,
        "part_present": part_present,
        "part_correct": part_correct,
        "records_present": records_present,
        "records_is_list": records_is_list,
        "records_evaluable": records_evaluable,
        "expected_record_count":
            EXPECTED_PART_COUNTS[part_number],
        "observed_record_count":
            observed_part_count,
        "record_count_matches":
            part_record_count_matches
    }

    if records_evaluable:
        combined_records.extend(records)

    if not valid_json:
        all_parts_json_valid = False

    if not records_evaluable:
        all_parts_records_evaluable = False

PART_EXECUTION_SUMMARY = {
    str(part_number): result
    for part_number, result
    in part_parsing_results.items()
}

PART_EXECUTION_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D6_branch_B_part_execution_summary.json"
)

PART_EXECUTION_SUMMARY_PATH.write_text(
    json.dumps(
        PART_EXECUTION_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

observed_record_count = (
    len(combined_records)
    if all_parts_records_evaluable
    else None
)

print(json.dumps(
    PART_EXECUTION_SUMMARY,
    ensure_ascii=False,
    indent=2
))
print("All parts JSON valid:", all_parts_json_valid)
print("All parts records evaluable:", all_parts_records_evaluable)
print("Combined observed records:", observed_record_count)


In [ ]:
# ============================================================
# 11. Validatation of combined record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

extracted_records = (
    combined_records
    if all_parts_records_evaluable
    else []
)

for record_index, record in enumerate(extracted_records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(
        expected_fields - actual_fields
    )
    extra_fields = sorted(
        actual_fields - expected_fields
    )

    if missing_fields or extra_fields:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields
        })

    for field in STRING_OR_NULL_FIELDS:
        value = record.get(field)
        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__
            })

    for field in NUMERIC_OR_NULL_FIELDS:
        value = record.get(field)
        if (
            value is not None
            and (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
            )
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__,
                "observed_value": value
            })

    missing_content = [
        field
        for field in MANDATORY_CONTENT_FIELDS
        if record.get(field) is None
    ]

    if missing_content:
        missing_mandatory_values.append({
            "record_index": record_index,
            "missing_mandatory_fields": missing_content
        })

record_schema_valid = (
    len(record_structure_issues) == 0
    if all_parts_records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if all_parts_records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if all_parts_records_evaluable
    else None
)

print("Record structure issues:", len(record_structure_issues))
print("Field type issues:", len(field_type_issues))
print("Records with missing mandatory fields:", len(missing_mandatory_values))


In [ ]:
# ============================================================
# 12. Content/scope diagnostics kept separate from schema validity
# ============================================================

if all_parts_records_evaluable:
    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    record_count_valid = (
        len(extracted_records) == EXPECTED_RECORD_COUNT
    )

    category_counts_valid = (
        observed_category_counts == EXPECTED_CATEGORY_COUNTS
    )

    duplicate_key_fields = [
        "Category",
        "Statement or Section",
        "Metric",
        "Business Area",
        "Reporting Period",
        "Source Location"
    ]

    duplicate_counter = Counter(
        tuple(record.get(field) for field in duplicate_key_fields)
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_record_keys = [
        list(key)
        for key, count in duplicate_counter.items()
        if count > 1
    ]

    duplicate_record_key_count = len(
        duplicate_record_keys
    )

else:
    observed_category_counts = None
    record_count_valid = None
    category_counts_valid = None
    duplicate_record_keys = None
    duplicate_record_key_count = None

CONTENT_DIAGNOSTICS = {
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count":
        len(extracted_records)
        if all_parts_records_evaluable
        else None,
    "record_count_matches_reference":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "category_counts_match_reference":
        category_counts_valid,
    "expected_part_counts":
        EXPECTED_PART_COUNTS,
    "part_execution_results":
        PART_EXECUTION_SUMMARY,
    "duplicate_record_key_count":
        duplicate_record_key_count,
    "mandatory_fields_complete":
        mandatory_fields_complete
}

print(json.dumps(
    CONTENT_DIAGNOSTICS,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 13. Post-extraction technical diagnostics
# ============================================================

structurally_evaluable = all([
    all_parts_json_valid,
    all_parts_records_evaluable,
    record_schema_valid is True,
    field_types_valid is True
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "all_parts_json_valid":
        bool(all_parts_json_valid),

    "all_parts_records_evaluable":
        bool(all_parts_records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "records_with_structure_issues":
        len(record_structure_issues)
        if all_parts_records_evaluable
        else None,

    "record_structure_issues":
        record_structure_issues
        if all_parts_records_evaluable
        else None,

    "records_with_type_issues":
        len({
            issue["record_index"]
            for issue in field_type_issues
        })
        if all_parts_records_evaluable
        else None,

    "field_type_issues":
        field_type_issues
        if all_parts_records_evaluable
        else None,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_values":
        missing_mandatory_values
        if all_parts_records_evaluable
        else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D6_branch_B_technical_diagnostics.json"
)

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    TECHNICAL_DIAGNOSTICS,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 14. Preservation combined parsed extraction only if all parts are evaluable
# ============================================================

COMBINED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D6_branch_B_combined_parsed_extraction.json"
)

combined_extraction_created = False
combined_extraction_sha256 = None

if structurally_evaluable:
    COMBINED_EXTRACTION = {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH,
        "records": extracted_records
    }

    COMBINED_EXTRACTION_PATH.write_text(
        json.dumps(
            COMBINED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    combined_extraction_created = True
    combined_extraction_sha256 = sha256_file(
        COMBINED_EXTRACTION_PATH
    )

    print("Combined parsed extraction saved.")
else:
    print(
        "Combined parsed extraction was not created because "
        "one or more parts are not evaluable."
    )


In [ ]:
# ============================================================
# 15. Creation of final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_files":
        PART_RAW_RESPONSE_METADATA,
    "part_execution_summary_file":
        PART_EXECUTION_SUMMARY_PATH.name,
    "part_parsing_results":
        PART_EXECUTION_SUMMARY,
    "all_parts_json_valid":
        all_parts_json_valid,
    "all_parts_records_evaluable":
        all_parts_records_evaluable,
    "combined_parsed_extraction_file":
        COMBINED_EXTRACTION_PATH.name
        if combined_extraction_created
        else None,
    "combined_parsed_extraction_sha256":
        combined_extraction_sha256,
    "observed_record_count":
        len(extracted_records)
        if all_parts_records_evaluable
        else None,
    "observed_category_counts":
        observed_category_counts,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "structurally_evaluable":
        bool(structurally_evaluable),
    "content_validation_performed":
        False,
    "notes": (
        "Branch B uses the same complete converted 10-page Markdown "
        "representation in all five predefined extraction runs. "
        "Only the extraction scope changes between parts, matching "
        "the five-part Branch A execution protocol. Stage 1 expected "
        "counts are retained only for post-extraction diagnostics and "
        "are not disclosed to the model. Content-level validation is "
        "performed separately in Validation B — D6."
    )
}

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D6_branch_B_experiment_metadata.json"
)

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "conversion_method":
        CONVERSION_METHOD,
    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "same_complete_representation_used_for_all_parts": True,
    "split_extraction_applied": True,
    "split_part_count": 5,
    "all_parts_json_valid":
        bool(all_parts_json_valid),
    "all_parts_records_evaluable":
        bool(all_parts_records_evaluable),
    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        len(extracted_records)
        if all_parts_records_evaluable
        else None,
    "record_count_matches":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "category_counts_match":
        category_counts_valid,
    "structurally_evaluable":
        bool(structurally_evaluable),
    "scope_complete":
        record_count_valid,
    "combined_parsed_extraction_created":
        combined_extraction_created,
    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D6."
    )
}

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D6_branch_B_experiment_summary.json"
)

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_SUMMARY,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 16. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PART_EXECUTION_SUMMARY_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

GENERATED_OUTPUTS.extend(
    PART_PROMPT_PATHS[part_number]
    for part_number in sorted(PART_PROMPT_PATHS)
)

GENERATED_OUTPUTS.extend(
    PART_RAW_RESPONSE_PATHS[part_number]
    for part_number in sorted(PART_RAW_RESPONSE_PATHS)
)

if combined_extraction_created:
    GENERATED_OUTPUTS.append(
        COMBINED_EXTRACTION_PATH
    )

print("Generated D6 Branch B files:")
for output_path in GENERATED_OUTPUTS:
    print("-", output_path.name)

for output_path in GENERATED_OUTPUTS:
    files.download(output_path)
